# Local YOLO Training Template

This notebook mirrors the workflow of the Colab reference but keeps everything local: configure your dataset, launch Ultralytics YOLO training, review metrics, and run quick sanity-check predictions without relying on Google Drive or Colab-specific APIs.


## How to use this notebook

1. Make sure your dataset follows the YOLO folder layout (`train/valid/test` subfolders, each with `images/` and `labels/`).
2. Update the paths and class names in the **Configuration** cell below.
3. Run the cells in order: install dependencies (if needed), verify your environment, configure the dataset, train, inspect metrics, and run sample predictions.
4. All training artifacts (weights, plots, predictions) are stored under the `runs/` folder inside this project directory by default.


In [ ]:
# Optionally install/upgrade the core dependencies used in this notebook
%pip install -q ultralytics opencv-python pillow matplotlib pyyaml


In [ ]:
import platform
import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO

# --- Update these variables to match your dataset ---
DATA_ROOT = Path("furniture_dataset")          # Folder containing train/valid/test
EXPERIMENT_NAME = "furniture_condition_local"   # Will appear inside runs/
MODEL_ARCH = "yolov8n.pt"                       # Or yolov8s.pt, yolov11n.pt, etc.
CLASSES = [
    "chair_broken",
    "chair_wornout",
    "sofa_broken",
    "sofa_wornout",
    "table_broken",
    "table_wornout",
]
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Derived paths
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "valid"
TEST_DIR = DATA_ROOT / "test"
DATA_CONFIG_PATH = Path("local_data.yaml")
RUNS_DIR = Path("runs")

# Create a lightweight data.yaml for Ultralytics
DATA_CONFIG = {
    "path": str(DATA_ROOT.resolve()),
    "train": str(TRAIN_DIR.resolve()),
    "val": str(VAL_DIR.resolve()),
    "test": str(TEST_DIR.resolve()) if TEST_DIR.exists() else None,
    "nc": len(CLASSES),
    "names": CLASSES,
}

with open(DATA_CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.dump(DATA_CONFIG, f)

print("Configuration saved to:", DATA_CONFIG_PATH.resolve())
print("Classes:", CLASSES)
print("Using device:", DEVICE)


In [ ]:
from collections import Counter
from PIL import Image
import random

def list_images(folder: Path):
    if not folder.exists():
        return []
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
    files = []
    for ext in exts:
        files.extend(folder.glob(ext))
    return sorted(files)

train_images = list_images(TRAIN_DIR / "images")
val_images = list_images(VAL_DIR / "images")
print(f"Train images: {len(train_images)} | Val images: {len(val_images)}")

if train_images:
    sample = random.choice(train_images)
    print("Showing sample:", sample)
    display(Image.open(sample))
else:
    print("No training images were found — check DATA_ROOT.")


In [ ]:
model = YOLO(MODEL_ARCH)

results = model.train(
    data=str(DATA_CONFIG_PATH),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
    exist_ok=True,
    pretrained=True,
)

RUN_DIR = Path(results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print(f"Training complete. Artifacts saved to: {RUN_DIR}")
print(f"Best weights: {BEST_WEIGHTS}")


In [ ]:
from IPython.display import Image as IPyImage, display

plot_files = [
    RUN_DIR / "results.png",
    RUN_DIR / "confusion_matrix.png",
    RUN_DIR / "F1_curve.png",
]

for plot in plot_files:
    if plot.exists():
        display(IPyImage(filename=plot))
    else:
        print(f"Plot not found yet: {plot}")


In [ ]:
trained_model = YOLO(str(BEST_WEIGHTS))

sample_pool = (val_images or train_images)[:3]
if not sample_pool:
    raise RuntimeError("No images available for inference preview. Add data to DATA_ROOT.")

PREDICT_SUBDIR = f"{EXPERIMENT_NAME}_predictions"

pred_results = trained_model.predict(
    source=[str(p) for p in sample_pool],
    conf=0.3,
    imgsz=IMAGE_SIZE,
    project=str(RUN_DIR.parent),
    name=PREDICT_SUBDIR,
    save=True,
    exist_ok=True,
)

print(f"Saved annotated predictions to: {RUN_DIR.parent / PREDICT_SUBDIR}")


## Next steps

- Adjust `MODEL_ARCH`, `EPOCHS`, `IMAGE_SIZE`, and `BATCH_SIZE` to suit your hardware and dataset size.
- Replace the `CLASSES` list with your own labels (matching the order used in your YOLO annotations).
- Expand the sanity-check cell to visualize bounding boxes or run validation metrics as needed.
- When satisfied with the model, copy `BEST_WEIGHTS` into your inference scripts (e.g., `enhanced_detection.py`).
